In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torch.optim.lr_scheduler import CosineAnnealingLR

from src.cnn13 import CNN13
from src.train import train_mean_teacher, train_rlgssl, train_rlgssl_plus

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## **Dataset Loading**

In [ ]:
transform = transforms.Compose([
      transforms.ToTensor(),
])
# load the CIFAR-100 for train and test
train_dataset = datasets.CIFAR100(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.CIFAR100(root="./data", train=False, download=True, transform=transform)

Now, to reproduce the results from the original RLGSSL, we need to create two dataloaders:
- labeled dataloader: 10,000 labeled samples
- unlabeled dataloader: 40,000 unlabeled samples

In [ ]:
targets = np.array(train_dataset.targets)
num_classes = 100
labeled_per_class = 10_000 // num_classes

In [ ]:
labeled_indices = []
unlabeled_indices = []

for class_idx in range(num_classes):
    class_indices = np.where(targets == class_idx)[0]
    # shuffle class indices for reproducibly
    np.random.shuffle(class_indices)

    # first 100 labeled
    labeled_class_indices = class_indices[:labeled_per_class]
    labeled_indices.extend(labeled_class_indices)
    # remaining 400 unlabeled
    unlabeled_class_indices = class_indices[labeled_per_class:]
    unlabeled_indices.extend(unlabeled_class_indices)

labeled_indices = np.array(labeled_indices)
unlabeled_indices = np.array(unlabeled_indices)

print(f"Labeled Set Size: {len(labeled_indices)}")
print(f"Unlabeled Set Size: {len(unlabeled_indices)}")

In [ ]:
labeled_dataset = Subset(train_dataset, labeled_indices)
unlabeled_dataset = Subset(train_dataset, unlabeled_indices)

In [ ]:
batch_size = 130
ratio = len(unlabeled_dataset) // len(labeled_dataset)
batch_size_labeled = batch_size // (ratio + 1)
batch_size_unlabeled = batch_size - batch_size_labeled

print(ratio)
print(f"Labeled Batch Size: {batch_size_labeled}")
print(f"Unlabeled Batch Size: {batch_size_unlabeled}")

In [ ]:
labeled_loader = DataLoader(labeled_dataset, batch_size=batch_size_labeled, shuffle=True, num_workers=0)
unlabeled_loader = DataLoader(unlabeled_dataset, batch_size=batch_size_unlabeled, shuffle=True, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Labeled Dataloader Size: {len(labeled_loader)}")
print(f"Unlabeled Dataloader Size: {len(unlabeled_loader)}")

## **Pretrain**

In [ ]:
import math

def cosine_annealing(epoch, max_epochs, lambda_min=0.0, lambda_max=1.0):
    if max_epochs <= 1:
        return lambda_max
    cos_term = math.cos(math.pi * epoch / (max_epochs - 1))
    return lambda_min + 0.5 * (lambda_max - lambda_min) * (1 - cos_term)

In [ ]:
epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student = CNN13().to(device)
teacher = CNN13().to(device)

optimizer = torch.optim.SGD(
    params=student.parameters(),
    lr=0.01,
    momentum=0.9,
    nesterov=True,
)

total_losses = []
sup_losses = []
cons_losses = []

for epoch in range(epochs):
    lambda_cons = cosine_annealing(epoch, epochs, lambda_min=0.0, lambda_max=1.0)
    losses = train_mean_teacher(labeled_loader, unlabeled_loader, student, teacher, optimizer, device, lambda_cons=lambda_cons)
    
    total_loss = losses["total_loss"]
    sup_loss = losses["sup_loss"]
    cons_loss = losses["cons_loss"]

    print(f"Train Loss: {total_loss:.4f}, Supervision Loss {sup_loss:.4f}, Consistency Loss {cons_loss:.4f}")
    print()

    total_losses.append(total_loss)
    sup_losses.append(sup_loss)
    cons_losses.append(cons_loss)

In [ ]:
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, total_losses, label="Total Loss", linewidth=2)
plt.plot(epochs_range, sup_losses, label="Supervision Loss", linewidth=2)
plt.plot(epochs_range, cons_losses, label="Consistency Loss", linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## **RLGSSL Train**

In [ ]:
epochs = 400

optimizer = torch.optim.SGD(
    params=student.parameters(),
    lr=0.1,
    momentum=0.9,
    nesterov=True,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,       
    eta_min=0,
)

total_losses = []
rl_losses = []
sup_losses = []
cons_losses = []

for epoch in range(epochs):
    losses = train_rlgssl(labeled_loader, unlabeled_loader, student, teacher, optimizer, device)
    
    total_loss = losses["total_loss"]
    rl_loss = losses["rl_loss"]
    sup_loss = losses["sup_loss"]
    cons_loss = losses["cons_loss"]

    print(f"EPOCH {epoch + 1} ---> Train Loss: {total_loss:.4f}, RL Loss: {rl_loss:.4f}, Supervision Loss {sup_loss:.4f}, Consistency Loss {cons_loss:.4f}")
    print()

    total_losses.append(total_loss)
    rl_losses.append(rl_loss)
    sup_losses.append(sup_loss)
    cons_losses.append(cons_loss)

In [ ]:
epochs = 200

optimizer = torch.optim.SGD(
    params=student.parameters(),
    lr=0.1,
    momentum=0.9,
    nesterov=True,
    weight_decay=1e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=epochs,       
    eta_min=0,
)

total_losses = []
rl_losses = []
sup_losses = []
cons_losses = []

for epoch in range(epochs):
    losses = train_rlgssl(labeled_loader, unlabeled_loader, student, teacher, optimizer, device)
    
    total_loss = losses["total_loss"]
    rl_loss = losses["rl_loss"]
    sup_loss = losses["sup_loss"]
    cons_loss = losses["cons_loss"]
    
    if (epoch + 1) % 10 == 0:
        print(f"EPOCH {epoch + 1}: ---> Train Loss: {total_loss:.4f}, RL Loss: {rl_loss:.4f}, Supervision Loss {sup_loss:.4f}, Consistency Loss {cons_loss:.4f}")
        print()

    total_losses.append(total_loss)
    rl_losses.append(rl_loss)
    sup_losses.append(sup_loss)
    cons_losses.append(cons_loss)

In [ ]:
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, total_losses, label="Total Loss", linewidth=2)
plt.plot(epochs_range, rl_losses, label="RL Loss", linewidth=2)
plt.plot(epochs_range, sup_losses, label="Supervision Loss", linewidth=2)
plt.plot(epochs_range, cons_losses, label="Consistency Loss", linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def evaluate_test_error(model, test_loader, device):
    model.eval()
    correct = 0
    total = 0

    for images, labels in test_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        logits = model(images)
        preds = logits.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

    accuracy = 100.0 * correct / total
    error = 100.0 - accuracy
    return error, accuracy

In [ ]:
test_error, test_acc = evaluate_test_error(student, test_loader, device)
print(f"Test Error: {test_error:.2f}% | Test Accuracy: {test_acc:.2f}%")

## **RLGSSL Plus Train**

In [ ]:
epochs = 50
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
student = CNN13().to(device)
teacher = CNN13().to(device)

optimizer = torch.optim.SGD(
    params=student.parameters(),
    lr=0.001,
    momentum=0.9,
    nesterov=True,
    weight_decay=1e-4,
)

total_losses = []
sup_losses = []
cons_losses = []

for epoch in range(epochs):
    lambda_cons = cosine_annealing(epoch, epochs, lambda_min=0.0, lambda_max=1.0)
    losses = train_mean_teacher(labeled_loader, unlabeled_loader, student, teacher, optimizer, device, lambda_cons=lambda_cons)
    
    total_loss = losses["total_loss"]
    sup_loss = losses["sup_loss"]
    cons_loss = losses["cons_loss"]

    if (epoch + 1) % 10 == 0:
        print(f"EPOCH {epoch + 1}: Train Loss: {total_loss:.4f}, Supervision Loss {sup_loss:.4f}, Consistency Loss {cons_loss:.4f}")
        print()

    total_losses.append(total_loss)
    sup_losses.append(sup_loss)
    cons_losses.append(cons_loss)

In [ ]:
epochs_range = range(1, epochs + 1)

plt.figure(figsize=(8, 5))
plt.plot(epochs_range, total_losses, label="Total Loss", linewidth=2)
plt.plot(epochs_range, rl_losses, label="RL Loss", linewidth=2)
plt.plot(epochs_range, sup_losses, label="Supervision Loss", linewidth=2)
plt.plot(epochs_range, cons_losses, label="Consistency Loss", linewidth=2)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Losses")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
test_error, test_acc = evaluate_test_error(student, test_loader, device)
print(f"Test Error: {test_error:.2f}% | Test Accuracy: {test_acc:.2f}%")